# Experiment 007 — B5 partial CLIP fine-tuning
Run every cell in order. Final-test data must remain sealed.

In [ ]:
%pip install -q 'numpy>=1.26,<3' 'Pillow>=10,<13' open_clip_torch==2.24.0 opencv-python-headless==4.11.0.86 'segment-anything @ git+https://github.com/facebookresearch/segment-anything.git@dca509fe793f601edb92606367a655c15ac00fdf'

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d /content/food-nutrition/.git ] || git clone --branch 'merge-A&B' --single-branch https://github.com/xychen5165/ECE539-Grp5-FoodNutritionEst.git /content/food-nutrition
!git -C /content/food-nutrition pull --ff-only origin 'merge-A&B'
!git -C /content/food-nutrition status --short
!git -C /content/food-nutrition rev-parse HEAD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
RUN_ROOT = Path('/content/drive/MyDrive/ECE539-FoodNutritionEst/005-b-input-comparison')
os.environ['RUN_ROOT'] = str(RUN_ROOT)
for required in ['clip_original.npz', 'clip_crop.npz', 'results/B4/metrics.json']:
    assert (RUN_ROOT / required).is_file(), required
print(RUN_ROOT)

In [ ]:
!python /content/food-nutrition/B_nutrition_estimation/capture_finetune_environment.py --output "$RUN_ROOT/environment-007.json" --run-root "$RUN_ROOT"

In [ ]:
!mkdir -p /content/b5-data/images /content/b5-data/crops
!rsync -a "$RUN_ROOT/images/" /content/b5-data/images/
!rsync -a "$RUN_ROOT/crops/" /content/b5-data/crops/
!find /content/b5-data/images -name rgb.png | wc -l
!find /content/b5-data/crops -name crop_rgb.png | wc -l

In [ ]:
!python /content/food-nutrition/B_nutrition_estimation/train_clip_finetune.py --image-root /content/b5-data --artifact-root "$RUN_ROOT" --output-root "$RUN_ROOT/results" --clip-cache "$RUN_ROOT/checkpoints"

In [ ]:
!python /content/food-nutrition/B_nutrition_estimation/compare_finetune_condition.py --results "$RUN_ROOT/results" --output "$RUN_ROOT/comparison-007.json"
!python /content/food-nutrition/B_nutrition_estimation/package_finetune_results.py --run-root "$RUN_ROOT" --output "$RUN_ROOT/007-small-results.zip"
!ls -lh "$RUN_ROOT/007-small-results.zip"

Download `007-small-results.zip` from Drive and attach it to the experiment task.